# Auditoria e análise exploratória — SIH/SUS: insuficiência renal

**Objetivo:** validar a exportação TabNet para “Insuficiência renal” (categoria agregada N17–N19) e descrever internações por UF e mês em 2024–2025.

A extração bruta possui competências adicionais de 2023 e usa `-` em células. Este notebook preserva a distinção entre zero e ausência/supressão, exclui 2023 da análise-alvo e não interpreta os resultados como incidência de doença ou número de pessoas únicas.


## Fonte e escopo

Fonte: [TabNet DATASUS — morbidade hospitalar por local de internação](https://tabnet.datasus.gov.br/cgi/deftohtm.exe?sih/cnv/niuf.def). A consulta registrada selecionou UF, Ano/mês atendimento, Internações e Lista de Morbidade CID-10 “Insuficiência renal”. A categoria corresponde ao agregado N17–N19, não a DRC isolada (N18).


### Viabilidade do recorte solicitado (N18 e óbito)

O CSV local contém a tabulação agregada `Internações` para a categoria “Insuficiência renal” (N17–N19). Ele não tem colunas de diagnóstico principal nem de óbito. Na interface do TabNet há conteúdo “Óbitos”, mas a categoria de morbidade renal disponível continua agregada; esse resultado não isolaria N18. Portanto, não calcular N18 nem mortalidade hospitalar a partir deste CSV.

Para cumprir esse recorte, é necessário obter os microdados AIH-RD do DATASUS para 2024–2025 e validar no layout do arquivo as variáveis de diagnóstico principal e desfecho/óbito. A página oficial de [Produção Hospitalar do SIH/SUS](https://datasus.saude.gov.br/acesso-a-informacao/producao-hospitalar-sih-sus/) lista os Dados Consolidados AIH (RD) por local de internação a partir de 2008 e encaminha à [Transferência de Arquivos do DATASUS](https://datasus.saude.gov.br/transferencia-de-arquivos2/). Os 648 arquivos RD 2024–2025 estão presentes em `Dados/01_fontes_originais/SIH/SIH_RD_2024_2025_bruto/`. A extração/reconciliação N18 ainda não é feita por este notebook; use `05_AUDITORIA_SIH_DBC_N18.ipynb`, que compara os DBCs com o CSV preliminar e com a tabela N17–N19.


## 1. Leitura e metadados


In [1]:
from pathlib import Path
import csv, re

CANDIDATOS = [
    Path("../Dados/02_derivados_preliminares/SIH/sih_cnv_niuf231237179_0_58_181.csv"),
    Path("../Dados/sih_cnv_niuf231237179_0_58_181.csv"),
    Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados/sih_cnv_niuf231237179_0_58_181.csv"),
]
ARQUIVO = next((p for p in CANDIDATOS if p.exists()), CANDIDATOS[0])
assert ARQUIVO.exists(), f"CSV SIH não encontrado: {ARQUIVO}"

# O arquivo TabNet foi salvo em codificação ocidental; latin-1 preserva acentos.
with ARQUIVO.open(encoding="latin-1", newline="") as f:
    linhas = list(csv.reader(f, delimiter=";"))

# A exportação contém quatro linhas de metadados antes do cabeçalho tabular.
metadados = linhas[:4]
cabecalho = linhas[4]
dados = linhas[5:]
print("Arquivo:", ARQUIVO)
print("Metadados:")
for linha in metadados: print(" ".join(linha))
print("Colunas tabulares:", len(cabecalho), "| linhas após cabeçalho:", len(dados))


Arquivo: ../Dados/02_derivados_preliminares/SIH/sih_cnv_niuf231237179_0_58_181.csv
Metadados:
 Morbidade Hospitalar do SUS - por local de internação - Brasil
Internações por Unidade da Federação e Ano/mês atendimento
Lista Morb  CID-10: Insuficiência renal
Período:2024-2025
Colunas tabulares: 35 | linhas após cabeçalho: 33


## 1A. Verificar se o arquivo permite filtrar N18 e óbito

Esta verificação impede que uma tabulação N17–N19 seja rotulada como N18. Ela lista os campos tabulares e procura arquivos AIH-RD locais.


In [2]:
from pathlib import Path

campos = set(cabecalho)
campos_n18 = {"DIAG_PRINC", "DIAG_PRINCIPAL"} & campos
campos_obito = {"MORTE", "OBITO", "ÓBITOS", "Óbitos"} & campos
pastas_dados = [Path("../Dados"), Path("estatistica-aplicada-a-analise-de-dados-40h/Trabalho-final/Dados")]
arquivos_rd = [
    arquivo
    for pasta in pastas_dados if pasta.exists()
    for arquivo in pasta.rglob("*.dbc")
]
print("Campos tabulares no CSV:", cabecalho)
print("Campo de diagnóstico principal encontrado:", bool(campos_n18))
print("Campo de óbito encontrado:", bool(campos_obito))
print("Arquivos AIH-RD locais (.dbc/.dbf):", len(arquivos_rd))
assert not campos_n18 and not campos_obito, "A fonte mudou; reavalie o método antes de continuar."
if arquivos_rd:
    print("Os DBCs estão presentes; veja o notebook 05_AUDITORIA_SIH_DBC_N18.ipynb para a reextração e a reconciliação.")
else:
    print("Bloqueio: os DBCs SIH-RD não foram encontrados nas pastas verificadas.")


Campos tabulares no CSV: ['Unidade da Federação', '2023/Jan', '2023/Abr', '2023/Mai', '2023/Jul', '2023/Ago', '2023/Set', '2023/Out', '2023/Nov', '2023/Dez', '2024/Jan', '2024/Fev', '2024/Mar', '2024/Abr', '2024/Mai', '2024/Jun', '2024/Jul', '2024/Ago', '2024/Set', '2024/Out', '2024/Nov', '2024/Dez', '2025/Jan', '2025/Fev', '2025/Mar', '2025/Abr', '2025/Mai', '2025/Jun', '2025/Jul', '2025/Ago', '2025/Set', '2025/Out', '2025/Nov', '2025/Dez', 'Total']
Campo de diagnóstico principal encontrado: False
Campo de óbito encontrado: False
Arquivos AIH-RD locais (.dbc/.dbf): 1296
Os DBCs estão presentes; veja o notebook 05_AUDITORIA_SIH_DBC_N18.ipynb para a reextração e a reconciliação.


## 2. Identificar meses e linhas de dados


In [3]:
meses_2024_2025 = [f"{ano}/{mes}" for ano in (2024, 2025) for mes in [
    "Jan", "Fev", "Mar", "Abr", "Mai", "Jun", "Jul", "Ago", "Set", "Out", "Nov", "Dez"
]]
meses_csv = cabecalho[1:-1]
meses_fora_escopo = [m for m in meses_csv if m not in meses_2024_2025]
assert cabecalho[0] == "Unidade da Federação" and cabecalho[-1] == "Total"
assert len(cabecalho) == len(set(cabecalho)), "Cabeçalhos repetidos."
assert len(meses_csv) == 33, f"Esperadas 33 competências visíveis na extração; vieram {len(meses_csv)}."
assert len(meses_fora_escopo) == 9, f"Esperavam-se 9 meses de 2023 adicionais; encontrados {len(meses_fora_escopo)}."
assert all(m.startswith("2023/") for m in meses_fora_escopo)

linhas_uf = [r for r in dados if re.match(r"^\s*\d{2}\s+", r[0] if r else "")]
assert len(linhas_uf) == 27, f"Esperadas 27 linhas de UF; encontradas {len(linhas_uf)}."
assert all(len(r) == len(cabecalho) for r in linhas_uf), "Largura irregular em linha estadual."
print("Competências-alvo:", len(meses_2024_2025))
print("Meses adicionais que não serão analisados:", meses_fora_escopo)


Competências-alvo: 24
Meses adicionais que não serão analisados: ['2023/Jan', '2023/Abr', '2023/Mai', '2023/Jul', '2023/Ago', '2023/Set', '2023/Out', '2023/Nov', '2023/Dez']


## 3. Tratar células e resumir 2024–2025


In [4]:
def parse_celula(valor):
    """Converte valor TabNet; '-' fica como ausente, nunca como zero automático."""
    valor = valor.strip()
    if valor in ("", "-"):
        return None
    numero = int(valor)
    if numero < 0: raise ValueError(f"Contagem negativa inesperada: {valor}")
    return numero

registros = []
for linha in linhas_uf:
    cod, nome = re.match(r"^\s*(\d{2})\s+(.+?)\s*$", linha[0]).groups()
    valores = dict(zip(meses_csv, [parse_celula(x) for x in linha[1:-1]]))
    registros.append({"codigo_uf":cod,"uf":nome,"valores":valores,"total_exportado":parse_celula(linha[-1])})

# Resume apenas o período solicitado e contabiliza ausências/supressões.
resumo_uf = []
for r in registros:
    valores = [r["valores"][m] for m in meses_2024_2025]
    observados = [v for v in valores if v is not None]
    resumo_uf.append({
        "codigo_uf":r["codigo_uf"], "uf":r["uf"],
        "internacoes_observadas":sum(observados),
        "meses_sem_valor":sum(v is None for v in valores),
        "media_mensal_observada":sum(observados)/len(observados) if observados else None,
    })

print("UF | internações nas células observadas | meses sem valor")
for r in sorted(resumo_uf,key=lambda x:x["internacoes_observadas"],reverse=True):
    print(f"{r['uf']}: {r['internacoes_observadas']:,} | {r['meses_sem_valor']}".replace(",","."))


UF | internações nas células observadas | meses sem valor
São Paulo: 68.026 | 0
Minas Gerais: 40.403 | 0
Rio de Janeiro: 26.241 | 0
Bahia: 21.018 | 0
Paraná: 20.362 | 0
Rio Grande do Sul: 20.159 | 0
Pernambuco: 13.251 | 0
Pará: 12.917 | 0
Santa Catarina: 11.104 | 0
Ceará: 9.935 | 0
Maranhão: 7.248 | 0
Goiás: 7.221 | 0
Espírito Santo: 7.040 | 0
Amazonas: 6.233 | 0
Distrito Federal: 5.535 | 0
Paraíba: 5.122 | 0
Alagoas: 4.540 | 0
Mato Grosso: 4.220 | 0
Mato Grosso do Sul: 4.202 | 0
Piauí: 3.347 | 0
Rio Grande do Norte: 2.838 | 0
Rondônia: 2.298 | 0
Tocantins: 2.051 | 0
Sergipe: 1.701 | 0
Roraima: 936 | 1
Amapá: 852 | 0
Acre: 774 | 0


## 4. Checagens de interpretação e próximos passos


In [5]:
ausencias = sum(r["meses_sem_valor"] for r in resumo_uf)
assert sum(len(r["valores"]) for r in registros) == 27 * 33
print(f"Células ausentes/suprimidas no recorte 2024–2025: {ausencias}")
print("Nota: os dados dos seis meses mais recentes estão sujeitos a atualização.")
print("A categoria é N17–N19 agregada; para DRC específica, extrair SIH com CID principal N18.")
print("A linha total da exportação não foi usada para validar os meses-alvo, pois agrega o período da consulta original.")


Células ausentes/suprimidas no recorte 2024–2025: 1
Nota: os dados dos seis meses mais recentes estão sujeitos a atualização.
A categoria é N17–N19 agregada; para DRC específica, extrair SIH com CID principal N18.
A linha total da exportação não foi usada para validar os meses-alvo, pois agrega o período da consulta original.


### Limitações

- “Internações” são eventos de internação registrados no SIH, não pacientes únicos.
- “Insuficiência renal” agrega N17–N19; não isola DRC (N18), etiologia nem diagnóstico principal.
- Hífen é preservado como ausente/suprimido. Somar células observadas não equivale necessariamente a total completo quando há células ausentes.
- A exportação contém nove meses de 2023 adicionais; eles são excluídos da análise solicitada.
- Dados recentes podem sofrer revisão. Para comparação temporal, registre a data da extração e atualize a consulta.


## 4. Gerar cópia analítica 2024–2025

A célula seguinte grava um arquivo separado em `Dados`, sem modificar a exportação bruta. A coluna total representa a soma das células observadas; hífens ficam em branco e são contados em `Meses ausentes`.


In [6]:
from pathlib import Path
import csv

CAMINHOS_SAIDA = [
    Path("../Dados/02_derivados_preliminares/SIH/sih_insuficiencia_renal_n17_n19_uf_mes_2024_2025_analitico.csv"),
]
SAIDA = next((p for p in CAMINHOS_SAIDA if p.parent.exists()), CAMINHOS_SAIDA[0])
SAIDA.parent.mkdir(parents=True, exist_ok=True)

cabecalho_analitico = ["Código UF", "UF", *meses_2024_2025,
                       "Total observado 2024-2025", "Meses ausentes"]
linhas_analiticas = []
for r in registros:
    valores = [r["valores"][mes] for mes in meses_2024_2025]
    observados = [v for v in valores if v is not None]
    linhas_analiticas.append([
        r["codigo_uf"], r["uf"],
        *["" if v is None else v for v in valores],
        sum(observados), sum(v is None for v in valores),
    ])

# utf-8-sig facilita abertura direta em ferramentas de planilha.
with SAIDA.open("w", encoding="utf-8-sig", newline="") as f:
    escritor = csv.writer(f, delimiter=";")
    escritor.writerow(cabecalho_analitico)
    escritor.writerows(linhas_analiticas)

# Reabre a cópia e verifica dimensões, unicidade e preservação da ausência.
with SAIDA.open(encoding="utf-8-sig", newline="") as f:
    verificacao = list(csv.reader(f, delimiter=";"))
assert len(verificacao) == 28, "A cópia deve ter cabeçalho e 27 UFs."
assert all(len(linha) == len(cabecalho_analitico) for linha in verificacao)
assert len({linha[0] for linha in verificacao[1:]}) == 27
assert sum(int(linha[-1]) for linha in verificacao[1:]) == sum(
    sum(celula == "" for celula in linha[2:26]) for linha in verificacao[1:]
)
assert sum(int(linha[-2]) for linha in verificacao[1:]) == sum(
    int(celula) for linha in verificacao[1:] for celula in linha[2:26] if celula != ""
)
print("Cópia analítica criada:", SAIDA)
print("UFs:", len(verificacao)-1, "| competências:", len(meses_2024_2025))
print("Total de internações observadas:", sum(int(linha[-2]) for linha in verificacao[1:]))
print("Meses ausentes:", sum(int(linha[-1]) for linha in verificacao[1:]))
print("Arquivo bruto preservado em:", ARQUIVO)


Cópia analítica criada: ../Dados/02_derivados_preliminares/SIH/sih_insuficiencia_renal_n17_n19_uf_mes_2024_2025_analitico.csv
UFs: 27 | competências: 24
Total de internações observadas: 309574
Meses ausentes: 1
Arquivo bruto preservado em: ../Dados/02_derivados_preliminares/SIH/sih_cnv_niuf231237179_0_58_181.csv


### Nota sobre a cópia analítica

O total `Total observado 2024-2025` exclui células ausentes/suprimidas; não deve ser tratado como total completo quando houver meses ausentes. A cópia não inclui o total original do TabNet porque ele agrega o recorte da exportação bruta, que também contém competências de 2023.
